Phase 1

In [25]:
import pandas as pd

url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/gws/merged_gw.csv"
df = pd.read_csv(url)

print(df.head())

                     name position           team   xP  assists  bonus  bps  \
0             Femi Seriki      DEF  Sheffield Utd  0.5        0      0    0   
1        Jack Hinshelwood      MID       Brighton  1.5        0      0    0   
2            Jadon Sancho      MID        Man Utd  3.0        0      0    4   
3  Rhys Norrington-Davies      DEF  Sheffield Utd  0.1        0      0    0   
4           Vitaly Janelt      MID      Brentford  2.1        0      0    6   

   clean_sheets  creativity  element  ...  team_h_score  threat  total_points  \
0             0         0.0      653  ...             0     0.0             0   
1             0         0.0      621  ...             4     0.0             0   
2             0        11.3      397  ...             1     8.0             1   
3             0         0.0      487  ...             0     0.0             0   
4             0        11.5      105  ...             2    17.0             2   

   transfers_balance  transfers_in  tr

In [26]:
print(df.columns)

Index(['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps',
       'clean_sheets', 'creativity', 'element', 'expected_assists',
       'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team',
       'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
       'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score',
       'threat', 'total_points', 'transfers_balance', 'transfers_in',
       'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW'],
      dtype='str')


In [27]:
df = df.sort_values(["name", "GW"])

group = df.groupby("name")

# Lag features (previous week)
df["lag_points"] = group["total_points"].shift(1)
df["lag_minutes"] = group["minutes"].shift(1)

# Rolling averages (last 3 games)
df["avg_points_3"] = group["total_points"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_minutes_3"] = group["minutes"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_goals_3"] = group["goals_scored"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df["avg_assists_3"] = group["assists"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)

In [28]:
feature_cols = [
    "value",
    "was_home",
    "avg_points_3",
    "avg_minutes_3",
    "avg_goals_3",
    "avg_assists_3",
    "expected_goals",
    "expected_assists",
    "xP"
]

target_col = "total_points"

In [29]:
model_df = df.dropna(subset=feature_cols + [target_col]).copy()

X = model_df[feature_cols]
y = model_df[target_col]

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 80% train, 20% test
    random_state=42  # ensures same split every run
)

In [31]:
from sklearn.linear_model import LinearRegression

# Initialize model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("LR MAE:", mean_absolute_error(y_test, lr_pred))
print("LR RMSE:", mean_squared_error(y_test, lr_pred) ** 0.5)

LR MAE: 0.8157581311268588
LR RMSE: 1.5611954902462677


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)

MAE: 0.709240203823028
RMSE: 1.5162953414049585


In [33]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

print("GB MAE:", mean_absolute_error(y_test, gb_pred))
print("GB RMSE:", mean_squared_error(y_test, gb_pred) ** 0.5)

GB MAE: 0.7182249368294853
GB RMSE: 1.509433074158103


In [34]:
def predict_points(features):
    return model.predict(features.reshape(1, -1))[0]